Представим себе, что мы хотим открыть музыкальный магазин и заведовать всеми его делами. Напишем программу, которая будет визуализировать наши 
мечты. Наверное, в первую очередь нам понадобится класс для самого магазина. У магазина есть счет, склад, возможность продавать инструменты и 
покупать их (может быть, что-то еще - что придумаете). Потом, нужен, конечно, класс для музыкального инструмента. Попробуйте реализовать музыкальные 
инструменты с помощью наследования: можно создать некий абстрактный класс "инструмент", а от него будут наследовать классы "гитара", "пианино" и так 
далее - при этом у гитар и пианино могут быть свои разные характеристики, н-р, у гитары количество струн, тип порожка и так далее. Можно и более 
подробную иерархию, если будете успевать (например, инструменты бывают электрическими и акустическими). Наконец, можно создать класс "покупатель", 
у которого будет кошелек.

Наконец, неплохо бы не забыть написать некий код, который будет приводить наши классы в движение: имитировать рабочий день магазина (можно в цикле), 
с рандомной вероятностью посещение покупателей, давать нашему владельцу-пользователю возможность решить, не пора ли закупить новые инструменты или, 
может быть, затеять рекламную кампанию.

In [31]:
import random
import time
import json

In [32]:
class Instrument:
    def __init__(self, name, price, instrument_type):
        self.name = name
        self.price = price
        self.instrument_type = instrument_type
    
    def __str__(self):
        return f"{self.name} ({self.instrument_type}) — {self.price}₽"

class Guitar(Instrument):
    def __init__(self, name, price, strings=6, guitar_type="акустическая"):
        super().__init__(name, price, "гитара")
        self.strings = strings
        self.guitar_type = guitar_type
    
    def __str__(self):
        return f"{self.guitar_type} гитара {self.name}, {self.strings} струн — {self.price}₽"



class ElectricGuitar(Guitar):
    def __init__(self, name, price, strings=6, pickup_type="single coil"):
        super().__init__(name, price, strings, guitar_type="электрическая")
        self.pickup_type = pickup_type

    def __str__(self):
        return f"электрогитара {self.name}, {self.strings} струн, звукосниматель: {self.pickup_type} — {self.price}₽"


class Piano(Instrument):
    def __init__(self, name, price, keys=88):
        super().__init__(name, price, "пианино")
        self.keys = keys
    
    def __str__(self):
        return f"пианино {self.name}, клавиш: {self.keys} — {self.price}₽"


class Drum(Instrument):
    def __init__(self, name, price, pieces=5):
        super().__init__(name, price, "ударная установка")
        self.pieces = pieces
    
    def __str__(self):
        return f"барабаны {self.name}, частей: {self.pieces} — {self.price}₽"

In [33]:
class Buyer:
    def __init__(self, name, money):
        self.name = name
        self.money = money
    
    def buy(self, store):
        available = [inst for inst in store.sklad if inst.price <= self.money]
        if not available:
            print(f"{self.name} посмотрела цены и ушла — слишком дорого")
            return False
        instrument = random.choice(available)
        self.money -= instrument.price
        store.schet += instrument.price
        store.sklad.remove(instrument)
        print(f"{self.name} купила {instrument}")
        return True

In [34]:
class Store:
    def __init__(self, schet):
        self.schet = schet
        self.sklad = []
        self.advertising_level = 1.0
        self.smoke_breaks_taken = 0
    
    def buy_instrument(self, instrument):
        if self.schet >= instrument.price:
            self.schet -= instrument.price
            self.sklad.append(instrument)
            print(f"закупили {instrument}")
        else:
            print("недостаточно денег для закупки")
    
    def manual_restock(self):
        print("\nдоступные инструменты для закупки:")
        options = [
            Guitar("Y38C BK_М167795", 15000),
            ElectricGuitar("IBANEZ GIO GRG121DX-BKF", 17000, pickup_type="humbucker"),
            Piano("TESLER KB-6120", 25000),
            Drum("MusicHall Deep Bass", 30000)
        ]
        for i, opt in enumerate(options, 1):
            print(f"{i}. {opt}")
        choice = input("введите номер инструмента для закупки (или enter для отмены): ")
        if choice.isdigit() and 1 <= int(choice) <= len(options):
            self.buy_instrument(options[int(choice)-1])
        else:
            print("закупка отменена")
    
    def advertise(self):
        cost = 5000
        if self.schet >= cost:
            self.schet -= cost
            self.advertising_level += 0.5
            print("провели кампанию! приток покупателей увеличился")
        else:
            print("недостаточно денег для рекламы")
    
    def smoke_break(self):
        print("владелец вышел на перекур...")
        self.smoke_breaks_taken += 1
        time.sleep(1)
        print("перекур окончен, возвращаемся к работе")
    
    def show_sklad(self):
        if not self.sklad:
            print("склад пуст")
        else:
            print("на складе:")
            for inst in self.sklad:
                print("  ", inst)
    
    def save_res(self, filename="store_res.json"):
        res = {
            "schet": self.schet,
            "advertising_level": self.advertising_level,
            "smoke_breaks_taken": self.smoke_breaks_taken,
            "sklad": [
                {
                    "type": type(inst).__name__,
                    "name": inst.name,
                    "price": inst.price,
                    "extra": inst.__dict__
                } for inst in self.sklad
            ]
        }
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(res, f, ensure_ascii=False, indent=4)
        print("итоги дня сохранены")

    def load_res(self, filename="store_res.json"):
        try:
            with open(filename, "r", encoding="utf-8") as f:
                res = json.load(f)
            self.schet = res["schet"]
            self.advertising_level = res["advertising_level"]
            self.smoke_breaks_taken = res["smoke_breaks_taken"]
            self.sklad = []
            for inst_data in res["sklad"]:
                inst_type = inst_data["type"]
                name = inst_data["name"]
                price = inst_data["price"]
                extra = inst_data["extra"]
                if inst_type == "Guitar":
                    self.sklad.append(Guitar(name, price, extra.get("strings",6), extra.get("guitar_type","акустическая")))
                elif inst_type == "ElectricGuitar":
                    self.sklad.append(ElectricGuitar(name, price, extra.get("strings",6), extra.get("pickup_type","single coil")))
                elif inst_type == "Piano":
                    self.sklad.append(Piano(name, price, extra.get("keys",88)))
                elif inst_type == "Drum":
                    self.sklad.append(Drum(name, price, extra.get("pieces",5)))
            print("состояние магазина загружено")
        except FileNotFoundError:
            print("файл с итогами не найден")

    def day_summary(self):
        print("\n--- итоги рабочего дня ---")
        print(f"денег на счету: {self.schet}₽")
        print(f"перекуров за день: {self.smoke_breaks_taken}")
        print(f"уровень рекламы: {self.advertising_level}")
        self.show_sklad()
        self.save_res()

In [35]:
store = Store(schet=100000)
store.load_res()  

if not store.sklad:
    store.buy_instrument(Guitar("Y38C BK_М167795", 15000))
    store.buy_instrument(Piano("TESLER KB-6120", 25000))
    store.buy_instrument(Drum("MusicHall Deep Bass", 30000))

buyers = [
    Buyer("аня", 12000),
    Buyer("элина", 45000),
    Buyer("наташа", 20000),
    Buyer("катя", 40000),
]

print("\n--- начало дня ---\n")

for hour in range(10, 18):
    print(f"\nсейчас {hour}:00. что будем делать?")
    print("1 - ждать покупателей")
    print("2 - проведем рекламную кампанию (5000₽)")
    print("3 - проведем закупку")
    print("4 - перекурим")
    print("5 - чекнем склад")
    choice = input("ваш выбор: ").strip()

    if choice == "2":
        store.advertise()
    elif choice == "3":
        store.manual_restock()
    elif choice == "4":
        store.smoke_break()
    elif choice == "5":
        store.show_sklad()
        continue
    else:
        buyer_chance = 0.4 * store.advertising_level
        if random.random() < buyer_chance and store.sklad:
            buyer = random.choice(buyers)
            buyer.buy(store)
        else:
            print(f"никто не пришёл в {hour}:00.")
    
    time.sleep(0.5)

store.day_summary()


состояние магазина загружено

--- начало дня ---


сейчас 10:00. что будем делать?
1 - ждать покупателей
2 - проведем рекламную кампанию (5000₽)
3 - проведем закупку
4 - перекурим
5 - чекнем склад
никто не пришёл в 10:00.

сейчас 11:00. что будем делать?
1 - ждать покупателей
2 - проведем рекламную кампанию (5000₽)
3 - проведем закупку
4 - перекурим
5 - чекнем склад
аня посмотрела цены и ушла — слишком дорого

сейчас 12:00. что будем делать?
1 - ждать покупателей
2 - проведем рекламную кампанию (5000₽)
3 - проведем закупку
4 - перекурим
5 - чекнем склад
владелец вышел на перекур...
перекур окончен, возвращаемся к работе

сейчас 13:00. что будем делать?
1 - ждать покупателей
2 - проведем рекламную кампанию (5000₽)
3 - проведем закупку
4 - перекурим
5 - чекнем склад
элина купила барабаны MusicHall Deep Bass, частей: 5 — 30000₽

сейчас 14:00. что будем делать?
1 - ждать покупателей
2 - проведем рекламную кампанию (5000₽)
3 - проведем закупку
4 - перекурим
5 - чекнем склад
владелец вышел н

In [36]:
import random
import time

In [41]:
class City:
    def __init__(self, name):
        self.name = name
        self.happiness = 50    
        self.ecology = 50      
        self.education = 50    
        self.budget = 100000  
        self.buildings = []

    def add_building(self, building):
        self.buildings.append(building)
        self.happiness += building.happiness_effect
        self.ecology += building.ecology_effect
        self.education += building.education_effect
        self.budget += building.immediate_budget_effect

    def apply_buildings_effects(self):
        for building in self.buildings:
            building.apply_effect(self)

    def random_event(self):
        event_chance = random.random()
        if event_chance < 0.1:
            self.happiness -= 5
            self.ecology -= 10
            print("пожар в городе! счастье и экология снизились")
        elif event_chance < 0.2:
            self.happiness += 5
            self.budget -= 2000
            print("праздник в городе! счастье повысилось, бюджет немного уменьшился")
        elif event_chance < 0.3:
            self.happiness -= 3
            self.education -= 2
            print("эпидемия снизила счастье и образование жителей")
        elif event_chance < 0.35:
            self.budget += 10000
            print("экономический рост! бюджет увеличен на 10000₽")
        else:
            print("сегодня особых событий не произошло")

    def show_stats(self):
        print(f"\n--- город {self.name} ---")
        print(f"счастье: {self.happiness}")
        print(f"экология: {self.ecology}")
        print(f"образование: {self.education}")
        print(f"бюджет: {self.budget}")
        print("постройки:", [b.name for b in self.buildings])

In [38]:
class Building:
    def __init__(self, name, cost, happiness_effect=0, ecology_effect=0, education_effect=0, immediate_budget_effect=0):
        self.name = name
        self.cost = cost
        self.happiness_effect = happiness_effect
        self.ecology_effect = ecology_effect
        self.education_effect = education_effect
        self.immediate_budget_effect = immediate_budget_effect

    def apply_effect(self, city):
        pass

class House(Building):
    def __init__(self):
        super().__init__("жилой дом", cost=20000, happiness_effect=5)

class School(Building):
    def __init__(self):
        super().__init__("школа", cost=50000, education_effect=10)

class Hospital(Building):
    def __init__(self):
        super().__init__("больница", cost=70000, happiness_effect=5, immediate_budget_effect=-1000)

    def apply_effect(self, city):
        if random.random() < 0.5:
            city.happiness += 2
            city.ecology += 1
            print("больница провела медосмотр (добрый вечер): счастье жителей выросло!")

class Factory(Building):
    def __init__(self):
        super().__init__("завод", cost=80000, immediate_budget_effect=10000, happiness_effect=-2, ecology_effect=-5)

    def apply_effect(self, city):
        if random.random() < 0.2:
            city.ecology -= 5
            print("на заводе произошла авария, экология снизилась на 5")
        city.budget += 5000

class Farm(Building):
    def __init__(self):
        super().__init__("ферма", cost=60000, happiness_effect=3, ecology_effect=-2, immediate_budget_effect=5000)
        self.food_stock = 0

    def apply_effect(self, city):
        produced = random.randint(5, 15)
        self.food_stock += produced
        city.happiness += 1
        city.budget += 2000
        print(f"ферма произвела {produced} единиц еды. бюджет увеличен на 2000₽")

        if random.random() < 0.1:
            lost = random.randint(1, 5)
            self.food_stock -= lost
            city.happiness -= 1
            print(f"на ферме произошла болезнь животных! потеряно {lost} еды, счастье немного снизилось")

In [42]:
class Game:
    def __init__(self, city_name):
        self.city = City(city_name)
        self.turn = 1

    def play(self):
        print(f"\nдобро пожаловать! вы управляете городом {self.city.name}")
        while True:
            print(f"\n--- ход {self.turn} ---")
            self.city.show_stats()
            print("\nчто вы хотите сделать?")
            print("1 - построить жилой дом (20000₽)")
            print("2 - построить школу (50000₽)")
            print("3 - построить больницу (70000₽)")
            print("4 - построить завод (80000₽)")
            print("5 - построить ферму (60000₽)")
            print("6 - пропустить ход")
            print("0 - выйти из игры")
            
            choice = input("Ваш выбор: ").strip()
            
            if choice == "0":
                print("игра завершена")
                break
            elif choice == "1" and self.city.budget >= 20000:
                self.city.add_building(House())
                self.city.budget -= 20000
            elif choice == "2" and self.city.budget >= 50000:
                self.city.add_building(School())
                self.city.budget -= 50000
            elif choice == "3" and self.city.budget >= 70000:
                self.city.add_building(Hospital())
                self.city.budget -= 70000
            elif choice == "4" and self.city.budget >= 80000:
                self.city.add_building(Factory())
                self.city.budget -= 80000
            elif choice == "5" and self.city.budget >= 60000:
                self.city.add_building(Farm())
                self.city.budget -= 60000
            elif choice == "6":
                print("вы пропустили ход")
            else:
                print("недостаточно денег или неверный выбор")
            
         
            self.city.apply_buildings_effects()
            self.city.random_event()
            self.turn += 1
            time.sleep(0.5)

game = Game("городочек")
game.play()



добро пожаловать! вы управляете городом городочек

--- ход 1 ---

--- город городочек ---
счастье: 50
экология: 50
образование: 50
бюджет: 100000
постройки: []

что вы хотите сделать?
1 - построить жилой дом (20000₽)
2 - построить школу (50000₽)
3 - построить больницу (70000₽)
4 - построить завод (80000₽)
5 - построить ферму (60000₽)
6 - пропустить ход
0 - выйти из игры
ферма произвела 10 единиц еды. бюджет увеличен на 2000₽
праздник в городе! счастье повысилось, бюджет немного уменьшился

--- ход 2 ---

--- город городочек ---
счастье: 59
экология: 48
образование: 50
бюджет: 45000
постройки: ['ферма']

что вы хотите сделать?
1 - построить жилой дом (20000₽)
2 - построить школу (50000₽)
3 - построить больницу (70000₽)
4 - построить завод (80000₽)
5 - построить ферму (60000₽)
6 - пропустить ход
0 - выйти из игры
ферма произвела 15 единиц еды. бюджет увеличен на 2000₽
экономический рост! бюджет увеличен на 10000₽

--- ход 3 ---

--- город городочек ---
счастье: 65
экология: 48
образован